# SmartFruit · Predictive Loss · Modelo v0

**Implementación técnica interna:** v1.3  
**Objetivo:** evaluar si la información disponible al finalizar la primera muestra contiene señal útil para estimar `loss%` y apoyar al Decision Layer.

> En el entregable académico, esta es la **primera versión del componente predictivo**, por lo que se presenta como **Modelo v0**.  
> La etiqueta `v1.3` identifica únicamente la iteración técnica del notebook.



In [1]:
# ============================================================
# 0 · LIBRERÍAS Y CONFIGURACIÓN GENERAL
# ============================================================
MODO_DEMO = False
SEMILLA = 42
EXCLUIR_LOSS_NEGATIVO = True
TEST_SIZE = 0.25

import warnings, os, re, hashlib, subprocess, sys, json, zipfile
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    median_absolute_error, r2_score
)
import joblib

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

rng = np.random.default_rng(SEMILLA)
print("Predictive v1.3 configurado.")

import unicodedata
from difflib import SequenceMatcher


Predictive v1.3 configurado.


## 0.1 · Configuración real de Google Drive

El notebook está preparado para tu estructura actual:

```text
MyDrive/SMARTFRUIT/data/reportes.zip
MyDrive/SMARTFRUIT/data/Weight IN - OUT by lot - item.xlsx
```

Para la integración de la demo usará:

```text
MyDrive/SMARTFRUIT/integration/SmartFruit_reception_manifest.csv
```

y guardará artefactos del modelo en:

```text
MyDrive/SMARTFRUIT/models/
```


In [2]:
# ============================================================
# 0.1 · RUTAS REALES DEL PROYECTO EN DRIVE
# ============================================================
DRIVE_PROJECT_FOLDER = "SMARTFRUIT"

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive")
    PROJECT_DIR = DRIVE_ROOT / DRIVE_PROJECT_FOLDER
else:
    # Fallback para ejecución local.
    candidates = [
        Path.cwd() / DRIVE_PROJECT_FOLDER,
        Path("/mnt/data") / DRIVE_PROJECT_FOLDER,
        Path.cwd(),
        Path("/mnt/data"),
    ]
    PROJECT_DIR = next((p for p in candidates if p.exists()), Path.cwd())

DATA_DIR_DRIVE = PROJECT_DIR / "data"
INTEGRATION_DIR = PROJECT_DIR / "integration"
VISION_DIR = PROJECT_DIR / "vision"
MODEL_OUTPUT_DIR = PROJECT_DIR / "models"

# Solo se crean carpetas de integración/salida.
INTEGRATION_DIR.mkdir(parents=True, exist_ok=True)
VISION_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NOMBRE_ARCHIVO_QC = "reportes.zip"
NOMBRE_IMP = "Weight IN - OUT by lot - item.xlsx"
RECEPTION_MANIFEST_NAME = "SmartFruit_reception_manifest.csv"

QC_ARCHIVE_PATH = DATA_DIR_DRIVE / NOMBRE_ARCHIVO_QC
IMP_PATH_DRIVE = DATA_DIR_DRIVE / NOMBRE_IMP
RECEPTION_MANIFEST_PATH = INTEGRATION_DIR / RECEPTION_MANIFEST_NAME

print("PROJECT_DIR:", PROJECT_DIR)
print("QC ZIP:", QC_ARCHIVE_PATH)
print("IMP workbook:", IMP_PATH_DRIVE)
print("Reception manifest:", RECEPTION_MANIFEST_PATH)

print("\nVerificación Drive")
print("  SMARTFRUIT:", PROJECT_DIR.exists())
print("  data:", DATA_DIR_DRIVE.exists())
print("  reportes.zip:", QC_ARCHIVE_PATH.exists())
print("  IMP:", IMP_PATH_DRIVE.exists())


Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/SMARTFRUIT
QC ZIP: /content/drive/MyDrive/SMARTFRUIT/data/reportes.zip
IMP workbook: /content/drive/MyDrive/SMARTFRUIT/data/Weight IN - OUT by lot - item.xlsx
Reception manifest: /content/drive/MyDrive/SMARTFRUIT/integration/SmartFruit_reception_manifest.csv

Verificación Drive
  SMARTFRUIT: True
  data: False
  reportes.zip: False
  IMP: False


## 0.2 · Reception Manifest de la demo

El `reception_manifest` representa la información que SmartFruit ya conoce **después del scan y antes de Vision**.

Vision no debe inferir `lot_id`, `label_brand`, `country_origin`, `pallet_id` ni `size`.


In [3]:
def _find_file_with_fallback(primary_path, filename):
    """
    Busca primero la ruta oficial de Drive y luego fallbacks locales.
    """
    candidates = [
        Path(primary_path),
        Path.cwd() / filename,
        Path("/content") / filename,
        Path("/mnt/data") / filename,
    ]

    for p in candidates:
        if p.exists():
            return p

    return None


def _normalize_colname(c):
    c = str(c).replace("\ufeff", "").strip()
    c = re.sub(r"\s+", "_", c)
    return c.lower()


def load_reception_manifest(path):
    """
    Lee el manifest tolerando:
    - UTF-8 con o sin BOM
    - coma, punto y coma o tab como separador
    - espacios/BOM en encabezados
    """
    path = Path(path)

    # Intento 1: autodetección de separador
    try:
        df = pd.read_csv(
            path,
            dtype=str,
            encoding="utf-8-sig",
            sep=None,
            engine="python"
        )
    except Exception:
        df = None

    # Fallbacks explícitos
    if df is None or len(df.columns) == 1:
        best = None
        for sep in [",", ";", "\t"]:
            try:
                tmp = pd.read_csv(
                    path,
                    dtype=str,
                    encoding="utf-8-sig",
                    sep=sep
                )
                if best is None or len(tmp.columns) > len(best.columns):
                    best = tmp
            except Exception:
                pass

        if best is None:
            raise RuntimeError(
                f"No pude leer el reception manifest: {path}"
            )
        df = best

    df.columns = [_normalize_colname(c) for c in df.columns]
    df = df.fillna("")

    return df


manifest_path_found = _find_file_with_fallback(
    RECEPTION_MANIFEST_PATH,
    RECEPTION_MANIFEST_NAME
)

if manifest_path_found is not None:
    reception_manifest = load_reception_manifest(manifest_path_found)

    print("Reception manifest cargado desde:", manifest_path_found)
    print("Casos disponibles:", len(reception_manifest))
    print("Columnas detectadas:")
    print(reception_manifest.columns.tolist())

    required_manifest_cols = [
        "scan_code",
        "lot_id",
        "label_brand",
        "country_origin",
        "pallet_id",
        "size_declared",
        "expected_count_per_box",
    ]

    missing_manifest_cols = [
        c for c in required_manifest_cols
        if c not in reception_manifest.columns
    ]

    if missing_manifest_cols:
        raise ValueError(
            "El archivo se encontró, pero no tiene la estructura esperada.\n"
            f"Faltan: {missing_manifest_cols}\n"
            f"Columnas encontradas: {reception_manifest.columns.tolist()}\n\n"
            "Verifica que el archivo correcto sea "
            "SmartFruit_reception_manifest.csv."
        )

    print("\n✓ Estructura del manifest válida.")

    display(
        reception_manifest[
            required_manifest_cols
        ].head(10)
    )

else:
    reception_manifest = None
    print(
        "⚠ Reception manifest aún no encontrado.\n"
        f"Ruta esperada: {RECEPTION_MANIFEST_PATH}\n"
        "Esto NO impide entrenar el predictivo; "
        "solo impide ejecutar la demo de scan."
    )


⚠ Reception manifest aún no encontrado.
Ruta esperada: /content/drive/MyDrive/SMARTFRUIT/integration/SmartFruit_reception_manifest.csv
Esto NO impide entrenar el predictivo; solo impide ejecutar la demo de scan.


### Opcional · guardar el manifest en la carpeta oficial

Si subes el CSV manualmente a Colab una vez, esta función permite copiarlo a `SMARTFRUIT/integration/`.


In [4]:
def save_manifest_to_drive(local_manifest_path):
    src = Path(local_manifest_path)
    if not src.exists():
        raise FileNotFoundError(src)

    RECEPTION_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    RECEPTION_MANIFEST_PATH.write_bytes(src.read_bytes())

    print("Manifest guardado en:", RECEPTION_MANIFEST_PATH)
    return RECEPTION_MANIFEST_PATH

# Ejemplo:
# save_manifest_to_drive("/content/SmartFruit_reception_manifest.csv")


In [5]:
def buscar_archivo(nombre):
    """Fallback local; Drive se revisa explícitamente antes."""
    candidatos = [Path.cwd(), Path("/content"), Path("/mnt/data")]
    for base in candidatos:
        if not base.exists():
            continue
        p = base / nombre
        if p.exists():
            return p
        try:
            matches = list(base.rglob(nombre))
            if matches:
                return matches[0]
        except Exception:
            pass
    return None


def _qc_files_in(directory, recursive=True):
    if not Path(directory).exists():
        return []
    iterator = Path(directory).rglob("*") if recursive else Path(directory).glob("*")
    return [
        p for p in iterator
        if p.is_file()
        and p.suffix.lower() in {".xlsx", ".csv"}
        and p.name != NOMBRE_IMP
    ]


def buscar_directorio_reportes():
    """Busca reportes ya extraídos, primero en Drive y luego localmente."""
    candidatos = [
        DATA_DIR_DRIVE / "reportes",
        DATA_DIR_DRIVE / "reportesMD",
        DATA_DIR_DRIVE / "_smartfruit_data" / "reportes",
        DATA_DIR_DRIVE / "_smartfruit_data" / "reportesMD",
        DATA_DIR_DRIVE / "_smartfruit_data",
        Path.cwd() / "reportes",
        Path.cwd() / "reportesMD",
        Path("/mnt/data") / "reportes",
        Path("/mnt/data") / "reportesMD",
    ]

    for d in candidatos:
        files = _qc_files_in(d, recursive=True)
        if files:
            return d
    return None


def preparar_fuentes():
    """
    Devuelve el directorio con reportes QC.

    Prioridad:
    1) reportes ya extraídos en SMARTFRUIT/data/
    2) SMARTFRUIT/data/reportes.zip
    3) fallback local
    """
    d = buscar_directorio_reportes()
    if d is not None:
        print("Usando reportes QC ya extraídos:", d)
        return d

    # Archivo real de tu Drive.
    archive = QC_ARCHIVE_PATH if QC_ARCHIVE_PATH.exists() else buscar_archivo(NOMBRE_ARCHIVO_QC)

    if archive is None or not Path(archive).exists():
        raise FileNotFoundError(
            "No encontré reportes.zip. Ruta esperada: "
            f"{QC_ARCHIVE_PATH}"
        )

    archive = Path(archive)

    # En Drive dejamos la extracción junto a los datos.
    if archive == QC_ARCHIVE_PATH:
        out_dir = DATA_DIR_DRIVE / "_smartfruit_data"
    else:
        out_dir = Path.cwd() / "_smartfruit_data"

    out_dir.mkdir(parents=True, exist_ok=True)

    existing = _qc_files_in(out_dir, recursive=True)
    if not existing:
        print("Extrayendo:", archive)
        with zipfile.ZipFile(archive, "r") as z:
            z.extractall(out_dir)
    else:
        print("Extracción QC ya existente; no se repite.")

    qc_files = _qc_files_in(out_dir, recursive=True)
    if not qc_files:
        raise FileNotFoundError(
            f"Se extrajo {archive}, pero no encontré XLSX/CSV de QC."
        )

    # Elegir la carpeta con más reportes directos.
    counts = {}
    for p in qc_files:
        counts[p.parent] = counts.get(p.parent, 0) + 1
    best_dir = max(counts, key=counts.get)

    print(f"Directorio QC: {best_dir} · {counts[best_dir]} archivos directos")
    return best_dir


def _txt(x):
    return '' if pd.isna(x) else str(x).strip()


def _find_label_value(raw, label):
    target = label.lower()
    for r in range(raw.shape[0]):
        for c in range(raw.shape[1]):
            s = _txt(raw.iat[r, c])
            if s.lower().startswith(target):
                if ':' in s:
                    after = s.split(':', 1)[1].strip()
                    if after:
                        return after
                for cc in range(c + 1, min(raw.shape[1], c + 5)):
                    v = raw.iat[r, cc]
                    if pd.notna(v) and _txt(v):
                        return v
    return np.nan


def _find_header_row(raw):
    for r in range(raw.shape[0]):
        vals = [_txt(v).lower() for v in raw.iloc[r].tolist()]
        if any('pallet' in v for v in vals) and 'sample' in vals:
            return r
    return None


def _norm_col(s):
    s = _txt(s).replace('\n', ' ').lower()
    s = re.sub(r'[^a-z0-9]+', '_', s).strip('_')
    return s or 'unnamed'


def _norm_lot(v):
    if pd.isna(v):
        return np.nan
    try:
        return str(int(float(v)))
    except Exception:
        return str(v).strip()


def _norm_size(v):
    if pd.isna(v):
        return None
    s = str(v).upper().strip()
    if '10X1' in s or '1LB' in s:
        return 'KEY'
    m = re.search(r'(\d{2,3})', s)
    return m.group(1) if m else re.sub(r'[^A-Z0-9]', '', s)


def _mode_or_nan(s):
    s = s.dropna().astype(str)
    if len(s) == 0:
        return np.nan
    m = s.mode()
    return m.iloc[0] if len(m) else s.iloc[0]


def parse_qc_report(path):
    path = Path(path)
    if path.suffix.lower() == '.csv':
        raw = pd.read_csv(path, header=None, dtype=object)
    else:
        raw = pd.read_excel(path, sheet_name='Report', header=None, dtype=object)

    inspection_date = pd.to_datetime(_find_label_value(raw, 'Inspection Date'), errors='coerce')
    lot_id = _norm_lot(_find_label_value(raw, 'Trucco Lot Number'))
    country = _find_label_value(raw, 'Country Of Origin')

    level = np.nan
    for v in raw.to_numpy().ravel():
        if pd.isna(v):
            continue
        m = re.search(r'Level\s*Flag\s*:\s*([123])', str(v), flags=re.I)
        if m:
            level = int(m.group(1))
            break

    hdr = _find_header_row(raw)
    if hdr is None:
        raise ValueError(f"No se encontró encabezado de detalle en {path.name}")

    bases = [_norm_col(c) for c in raw.iloc[hdr].tolist()]
    counts, cols = {}, []
    for b in bases:
        counts[b] = counts.get(b, 0) + 1
        cols.append(b if counts[b] == 1 else f'{b}_{counts[b]}')

    tab = raw.iloc[hdr + 1:].copy()
    tab.columns = cols

    pallet_col = next((c for c in cols if 'pallet' in c), None)
    sample_col = next((c for c in cols if c == 'sample'), None)
    if pallet_col is None or sample_col is None:
        raise ValueError(f"No se encontraron columnas pallet/sample en {path.name}")

    sample_num = pd.to_numeric(tab[sample_col], errors='coerce')
    valid = tab[pd.notna(tab[pallet_col]) & sample_num.notna() & (sample_num > 0)].copy()

    def num(name):
        if name not in valid.columns:
            return pd.Series(np.nan, index=valid.index)
        return pd.to_numeric(valid[name], errors='coerce')

    out = pd.DataFrame({
        'source_file': path.name,
        'inspection_date': inspection_date,
        'lot_id': lot_id,
        'country_origin': country,
        'Level_Flag': level,
        'label_brand': valid.get('label'),
        'pallet_id': valid[pallet_col].astype(str).str.replace(r'\.0$', '', regex=True),
        'size': valid.get('size'),
        'avg_count_per_box': num('avg_count_per_box'),
        'plu_pct': num('plu_pct'),
        'sample_count': num('sample'),
        'open_rating': valid.get('open_rating'),
        'color_rating': valid.get('color_rating'),
        'pulp_temp': num('pulp'),
        'decay': num('decay'),
        'styler': num('styler'),
        'mold': num('mold'),
        'soft': num('soft'),
        'skin_breakdown': num('skin_breakdown'),
        'oleocellosis': num('oleocellosis'),
        'wounds': num('wounds'),
        'int_dryness': num('int_dryness'),
        'black_tips': num('black_tips'),
        'skin_defect': num('skin_defect'),
        'scars': num('scars'),
        'white_stain': num('white_stain'),
        'blanching': num('blanching'),
        'yellow_color': num('yellow_color'),
        'stain': num('stain'),
        'undersize': num('undersize'),
    })
    return out


In [6]:
DATA_DIR = preparar_fuentes()

IMP_PATH = IMP_PATH_DRIVE if IMP_PATH_DRIVE.exists() else DATA_DIR / NOMBRE_IMP
if not IMP_PATH.exists():
    IMP_PATH = buscar_archivo(NOMBRE_IMP)
if IMP_PATH is None or not Path(IMP_PATH).exists():
    raise FileNotFoundError(
        f"No encontré {NOMBRE_IMP}. Súbelo al mismo entorno del notebook."
    )
IMP_PATH = Path(IMP_PATH)

all_qc_files = sorted([
    p for p in DATA_DIR.rglob('*')
    if p.is_file()
    and p.suffix.lower() in {'.xlsx', '.csv'}
    and p.name != NOMBRE_IMP
])

print("Workbook IMP:", IMP_PATH)
print("Archivos QC encontrados:", len(all_qc_files))
if len(all_qc_files) == 0:
    raise FileNotFoundError(
        f"No se encontraron reportes QC dentro de {DATA_DIR}"
    )

seen, unique_qc, duplicados = {}, [], []
for p in all_qc_files:
    h = hashlib.sha256(p.read_bytes()).hexdigest()
    if h in seen:
        duplicados.append((p.name, seen[h].name))
    else:
        seen[h] = p
        unique_qc.append(p)

qc_rows_raw = pd.concat([parse_qc_report(p) for p in unique_qc], ignore_index=True)

mask_fuera_scope = (
    qc_rows_raw['label_brand'].astype(str).str.strip().str.lower().eq('generic') &
    qc_rows_raw['size'].astype(str).str.contains('10x1', case=False, na=False)
)
qc_rows = qc_rows_raw.loc[~mask_fuera_scope].copy()

DEFECTS = [
    'oleocellosis', 'styler', 'skin_breakdown',
    'decay', 'mold', 'scars', 'blanching', 'yellow_color'
]

for c in DEFECTS:
    qc_rows[c + '_pct'] = np.where(
        qc_rows[c].notna() & (qc_rows['sample_count'] > 0),
        qc_rows[c] / qc_rows['sample_count'] * 100,
        np.nan
    )

imp_loss = pd.read_excel(IMP_PATH, sheet_name='REPACK LOSS REPORT')
imp_loss = imp_loss.loc[:, ~imp_loss.columns.astype(str).str.startswith('Unnamed:')].copy()
imp_loss['DateReceived'] = pd.to_datetime(imp_loss['DateReceived'], errors='coerce')
imp_loss['lot_str'] = imp_loss['LotNumber'].map(_norm_lot)

wo = imp_loss.groupby('RepackOrderID').agg(
    loss_pct=('ORDERLossWeightPercentage', 'first'),
    initial_weight=('ORDERInitialWeight', 'first'),
    final_weight=('ORDERFinalWeight', 'first'),
    n_lots=('LotNumber', 'nunique'),
    date_received=('DateReceived', 'min')
).reset_index()

wo['loss_calc'] = (
    (wo['initial_weight'] - wo['final_weight']) / wo['initial_weight'] * 100
)

print(f"QC: {len(unique_qc)} reportes únicos | {qc_rows['lot_id'].nunique()} lotes | {len(qc_rows)} filas")
print(f"IMP: {len(wo)} Work Orders")
print(f"WO multi-lote: {(wo['n_lots'] > 1).sum()} / {len(wo)}")
print(f"WO de un solo lote: {(wo['n_lots'] == 1).sum()} / {len(wo)}")
print(f"Loss negativos: {(wo['loss_calc'] < 0).sum()}")


FileNotFoundError: No encontré reportes.zip. Ruta esperada: /content/drive/MyDrive/SMARTFRUIT/data/reportes.zip

## EDA / Data Quality antes del modelado

Esta sección se ejecuta **después de construir `qc_rows`** y antes del split/entrenamiento.

Se revisan:
- consistencia de `label_brand`;
- consistencia de `country_origin`;
- normalización de `size`;
- missingness;
- rangos y percentiles de variables numéricas;
- defectos mayores que `sample_count`;
- porcentajes fuera de `[0, 100]`;
- outliers descriptivos.

Se conserva siempre el valor original. No se corrigen silenciosamente categorías dudosas.


In [ ]:
# ============================================================
# EDA / DATA QUALITY
# ============================================================

if "qc_rows" not in globals():
    raise RuntimeError(
        "qc_rows no existe. Ejecuta el notebook desde el inicio."
    )

def normalize_text_basic(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    if not s:
        return np.nan
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.upper()
    s = (
        s.replace("’", "'")
         .replace("‘", "'")
         .replace("`", "'")
         .replace("´", "'")
    )
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalize_brand_key(value):
    s = normalize_text_basic(value)
    if pd.isna(s):
        return np.nan
    s = re.sub(r"[^A-Z0-9&]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


BRAND_ALIASES = {
    "DON VITO": "DON VITO",
    "DON VITO S": "DON VITO",
    "DON VITOS": "DON VITO",
}

BRAND_REVIEW_NOTES = {
    "RDL PRODDUCE": "Posible typo de RDL PRODUCE; confirmar antes de consolidar.",
}


# ---------- BRAND ----------
if "label_brand" not in qc_rows.columns:
    raise RuntimeError("No existe la columna label_brand en qc_rows.")

qc_rows["label_brand_raw"] = qc_rows["label_brand"]
qc_rows["label_brand_norm"] = qc_rows["label_brand_raw"].apply(normalize_brand_key)
qc_rows["label_brand_std"] = qc_rows["label_brand_norm"].replace(BRAND_ALIASES)


brand_summary = pd.DataFrame({
    "metric": [
        "brands_raw_unique",
        "brands_norm_unique",
        "brands_std_unique",
        "rows_with_brand",
        "rows_without_brand",
    ],
    "value": [
        qc_rows["label_brand_raw"].nunique(dropna=True),
        qc_rows["label_brand_norm"].nunique(dropna=True),
        qc_rows["label_brand_std"].nunique(dropna=True),
        int(qc_rows["label_brand_raw"].notna().sum()),
        int(qc_rows["label_brand_raw"].isna().sum()),
    ],
})

print("=== BRAND: resumen ===")
display(brand_summary)

brand_mapping_audit = (
    qc_rows
    .groupby(
        ["label_brand_raw", "label_brand_norm", "label_brand_std"],
        dropna=False
    )
    .agg(
        n_rows=("lot_id", "size") if "lot_id" in qc_rows.columns else ("label_brand_raw", "size"),
        n_lots=("lot_id", "nunique") if "lot_id" in qc_rows.columns else ("label_brand_raw", "size"),
    )
    .reset_index()
    .sort_values(["label_brand_std", "n_rows"], ascending=[True, False])
)

print("\n=== BRAND: mapping raw → std ===")
display(brand_mapping_audit)

brand_std_audit = (
    qc_rows
    .groupby("label_brand_std", dropna=False)
    .agg(
        n_rows=("label_brand_raw", "size"),
        n_lots=("lot_id", "nunique") if "lot_id" in qc_rows.columns else ("label_brand_raw", "size"),
        n_pallets=("pallet_id", "nunique") if "pallet_id" in qc_rows.columns else ("label_brand_raw", "size"),
    )
    .reset_index()
    .sort_values(["n_lots", "n_rows"], ascending=False)
)

print("\n=== BRAND: distribución estandarizada ===")
display(brand_std_audit.head(30))


# Sugerencias de similitud SOLO para revisión
_brands = sorted(
    qc_rows["label_brand_std"].dropna().astype(str).unique().tolist()
)

_review_pairs = []
for i, a in enumerate(_brands):
    for b in _brands[i+1:]:
        ratio = SequenceMatcher(None, a, b).ratio()
        if ratio >= 0.84 and a != b:
            _review_pairs.append({
                "brand_a": a,
                "brand_b": b,
                "similarity": round(ratio, 3),
            })

brand_similarity_review = (
    pd.DataFrame(_review_pairs).sort_values("similarity", ascending=False)
    if _review_pairs
    else pd.DataFrame(columns=["brand_a", "brand_b", "similarity"])
)

print("\n=== BRAND: nombres similares para revisión, sin auto-merge ===")
display(brand_similarity_review.head(30))

manual_brand_review = pd.DataFrame([
    {
        "normalized_value": key,
        "note": note,
        "observed": bool((qc_rows["label_brand_norm"] == key).any()),
    }
    for key, note in BRAND_REVIEW_NOTES.items()
])

print("\n=== BRAND: revisión manual pendiente ===")
display(manual_brand_review)


# ---------- COUNTRY ----------
if "country_origin" in qc_rows.columns:
    qc_rows["country_origin_raw"] = qc_rows["country_origin"]
    qc_rows["country_origin_std"] = (
        qc_rows["country_origin_raw"].apply(normalize_text_basic)
    )

    country_mapping = (
        qc_rows.groupby(
            ["country_origin_raw", "country_origin_std"],
            dropna=False
        )
        .agg(
            n_rows=("country_origin_raw", "size"),
            n_lots=("lot_id", "nunique") if "lot_id" in qc_rows.columns else ("country_origin_raw", "size"),
        )
        .reset_index()
        .sort_values(["n_lots", "n_rows"], ascending=False)
    )

    print("\n=== COUNTRY: raw → std ===")
    display(country_mapping)
else:
    qc_rows["country_origin_raw"] = np.nan
    qc_rows["country_origin_std"] = np.nan
    print("Advertencia: country_origin no está disponible.")


# ---------- SIZE ----------
if "size" in qc_rows.columns:
    qc_rows["size_raw"] = qc_rows["size"]

    if "_norm_size" in globals():
        qc_rows["size_norm"] = qc_rows["size_raw"].apply(_norm_size)
    else:
        # fallback conservador: extraer primer número entero reconocible
        def _fallback_norm_size(x):
            if pd.isna(x):
                return np.nan
            m = re.search(r"\d+", str(x))
            return int(m.group()) if m else np.nan

        qc_rows["size_norm"] = qc_rows["size_raw"].apply(_fallback_norm_size)

    size_mapping = (
        qc_rows.groupby(["size_raw", "size_norm"], dropna=False)
        .agg(
            n_rows=("size_raw", "size"),
            n_lots=("lot_id", "nunique") if "lot_id" in qc_rows.columns else ("size_raw", "size"),
        )
        .reset_index()
        .sort_values(["size_norm", "n_rows"])
    )

    print("\n=== SIZE: raw → norm ===")
    display(size_mapping.head(50))
else:
    qc_rows["size_raw"] = np.nan
    qc_rows["size_norm"] = np.nan
    print("Advertencia: size no está disponible.")


# ---------- MISSINGNESS ----------
_eda_cols = [
    c for c in [
        "label_brand_std",
        "country_origin_std",
        "size_norm",
        "sample_count",
        "avg_count_per_box",
        "pulp_temp",
        "oleocellosis",
        "styler",
        "skin_breakdown",
        "decay",
        "mold",
        "scars",
        "blanching",
        "yellow_color",
    ]
    if c in qc_rows.columns
]

missingness = pd.DataFrame({
    "column": _eda_cols,
    "missing_n": [qc_rows[c].isna().sum() for c in _eda_cols],
})
missingness["missing_pct"] = (
    missingness["missing_n"] / len(qc_rows) * 100
).round(2)

print("\n=== Missingness ===")
display(missingness.sort_values("missing_pct", ascending=False))


# ---------- NUMERICAL EDA ----------
_numeric_cols = [
    c for c in [
        "sample_count",
        "avg_count_per_box",
        "pulp_temp",
        "oleocellosis",
        "styler",
        "skin_breakdown",
        "decay",
        "mold",
        "scars",
        "blanching",
        "yellow_color",
    ]
    if c in qc_rows.columns
]

if _numeric_cols:
    _num_df = qc_rows[_numeric_cols].apply(pd.to_numeric, errors="coerce")

    numeric_eda = (
        _num_df
        .describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99])
        .T
    )
    numeric_eda["missing_n"] = _num_df.isna().sum()
    numeric_eda["missing_pct"] = (
        numeric_eda["missing_n"] / len(qc_rows) * 100
    ).round(2)

    print("\n=== EDA numérico ===")
    _cols_show = [
        c for c in [
            "count", "missing_n", "missing_pct",
            "min", "1%", "5%", "50%", "95%", "99%",
            "max", "mean", "std"
        ]
        if c in numeric_eda.columns
    ]
    display(numeric_eda[_cols_show].round(3))
else:
    numeric_eda = pd.DataFrame()
    print("No hay columnas numéricas esperadas para EDA.")


# ---------- CONSISTENCY CHECKS ----------
checks = []

sample_numeric = (
    pd.to_numeric(qc_rows["sample_count"], errors="coerce")
    if "sample_count" in qc_rows.columns
    else pd.Series(np.nan, index=qc_rows.index)
)

if "sample_count" in qc_rows.columns:
    checks.append({
        "check": "sample_count <= 0",
        "n_rows": int((sample_numeric <= 0).fillna(False).sum()),
    })

if "avg_count_per_box" in qc_rows.columns:
    _avg = pd.to_numeric(qc_rows["avg_count_per_box"], errors="coerce")
    checks.append({
        "check": "avg_count_per_box <= 0",
        "n_rows": int((_avg <= 0).fillna(False).sum()),
    })

DEFECT_COUNT_COLUMNS = [
    "oleocellosis",
    "styler",
    "skin_breakdown",
    "decay",
    "mold",
    "scars",
    "blanching",
    "yellow_color",
]

for defect in DEFECT_COUNT_COLUMNS:
    if defect not in qc_rows.columns:
        continue

    d = pd.to_numeric(qc_rows[defect], errors="coerce")

    checks.append({
        "check": f"{defect} < 0",
        "n_rows": int((d < 0).fillna(False).sum()),
    })

    if "sample_count" in qc_rows.columns:
        checks.append({
            "check": f"{defect} > sample_count",
            "n_rows": int(
                ((d > sample_numeric) & d.notna() & sample_numeric.notna()).sum()
            ),
        })

PCT_COLUMNS_EDA = [
    c for c in qc_rows.columns
    if c.endswith("_pct")
]

for pct_col in PCT_COLUMNS_EDA:
    p = pd.to_numeric(qc_rows[pct_col], errors="coerce")

    checks.append({
        "check": f"{pct_col} < 0",
        "n_rows": int((p < 0).fillna(False).sum()),
    })
    checks.append({
        "check": f"{pct_col} > 100",
        "n_rows": int((p > 100).fillna(False).sum()),
    })

consistency_df = pd.DataFrame(checks)

print("\n=== Reglas de consistencia ===")
display(consistency_df)


# ---------- OUTLIERS: reporte, no exclusión ----------
_outlier_rows = []

for col in _numeric_cols:
    s = pd.to_numeric(qc_rows[col], errors="coerce")
    valid = s.dropna()

    if len(valid) < 5:
        continue

    q1 = valid.quantile(0.25)
    q3 = valid.quantile(0.75)
    iqr = q3 - q1

    if pd.isna(iqr) or iqr == 0:
        continue

    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr

    _outlier_rows.append({
        "variable": col,
        "q1": q1,
        "q3": q3,
        "iqr_low": low,
        "iqr_high": high,
        "n_below": int((s < low).fillna(False).sum()),
        "n_above": int((s > high).fillna(False).sum()),
    })

outlier_audit = pd.DataFrame(_outlier_rows)

print("\n=== Outliers descriptivos por IQR ===")
print("Se reportan; NO se eliminan automáticamente.")
display(outlier_audit.round(3))


# ---------- RESUMEN ----------
eda_quality_summary = {
    "n_qc_rows": int(len(qc_rows)),
    "brands_raw_unique": int(qc_rows["label_brand_raw"].nunique(dropna=True)),
    "brands_std_unique": int(qc_rows["label_brand_std"].nunique(dropna=True)),
    "countries_raw_unique": int(qc_rows["country_origin_raw"].nunique(dropna=True)),
    "countries_std_unique": int(qc_rows["country_origin_std"].nunique(dropna=True)),
    "sizes_norm_unique": int(qc_rows["size_norm"].nunique(dropna=True)),
    "consistency_issues_total": (
        int(consistency_df["n_rows"].sum())
        if not consistency_df.empty else 0
    ),
}

print("\n=== Resumen EDA / Data Quality ===")
print(eda_quality_summary)


## Construcción de la tabla analítica v0

Usamos **solo el primer QC disponible de cada lote** y **solo Work Orders de un único lote**.  
Esto evita usar inspecciones posteriores y evita atribución ambigua en Work Orders multi-lote.


In [ ]:
first_date = qc_rows.groupby('lot_id')['inspection_date'].transform('min')
qc_first = qc_rows[qc_rows['inspection_date'].eq(first_date)].copy()
qc_first['size_norm'] = qc_first['size'].map(_norm_size)

def weighted_defect_pct(g, defect):
    valid = g[g[defect].notna() & g['sample_count'].notna() & (g['sample_count'] > 0)]
    if valid.empty:
        return np.nan
    return valid[defect].sum() / valid['sample_count'].sum() * 100

records = []
for lot_id, g in qc_first.groupby('lot_id'):
    rec = {
        'lot_id': str(lot_id),
        'inspection_date': g['inspection_date'].min(),
        'inspection_month': int(g['inspection_date'].min().month) if g['inspection_date'].notna().any() else np.nan,
        'country_origin_raw': _mode_or_nan(g['country_origin_raw']),
        'country_origin_std': _mode_or_nan(g['country_origin_std']),
        'label_brand_raw': _mode_or_nan(g['label_brand_raw']),
        'label_brand_std': _mode_or_nan(g['label_brand_std']),
        'size_norm': _mode_or_nan(g['size_norm']),
        'sample_count_total': g['sample_count'].sum(min_count=1),
        'n_pallets': g['pallet_id'].nunique(),
        'avg_count_per_box_mean': g['avg_count_per_box'].mean(),
        'avg_count_per_box_max': g['avg_count_per_box'].max(),
        'pulp_temp_mean': g['pulp_temp'].mean(),
        'pulp_temp_max': g['pulp_temp'].max(),
    }
    for d in DEFECTS:
        rec[f'{d}_pct'] = weighted_defect_pct(g, d)
        rec[f'{d}_pct_max_pallet'] = g[d + '_pct'].max()
    records.append(rec)

lot_features = pd.DataFrame(records)

wo_single_ids = set(wo.loc[wo['n_lots'].eq(1), 'RepackOrderID'])
single_map = (
    imp_loss[imp_loss['RepackOrderID'].isin(wo_single_ids)]
    .groupby('RepackOrderID')
    .agg(lot_id=('lot_str', 'first'))
    .reset_index()
)

model_df = (
    wo.loc[wo['n_lots'].eq(1), ['RepackOrderID', 'loss_calc', 'date_received']]
    .merge(single_map, on='RepackOrderID', how='left')
    .merge(lot_features, on='lot_id', how='inner')
)

model_df['target_valid'] = model_df['loss_calc'].notna()
if EXCLUIR_LOSS_NEGATIVO:
    model_df['target_valid'] &= model_df['loss_calc'].ge(0)

audit_df = model_df.copy()
model_df = model_df[model_df['target_valid']].copy()

print("WO single-lot con QC enlazado:", len(audit_df))
print("Usados en entrenamiento principal:", len(model_df))
print("Lotes únicos disponibles:", model_df['lot_id'].nunique())
print("Loss mediano:", round(model_df['loss_calc'].median(), 2))
display(model_df.head())


In [ ]:
summary = pd.DataFrame({
    'dtype': model_df.dtypes.astype(str),
    'missing_%': model_df.isna().mean().mul(100).round(1),
    'n_unique': model_df.nunique(dropna=True)
}).sort_values('missing_%', ascending=False)

display(summary)


## Auditoría de `label_brand`

`label_brand` se trata como **feature de contexto**, no como identificador del registro. Antes de interpretar su aporte se revisa cuántos lotes/WOs existen por brand. El encoder agrupa categorías demasiado infrecuentes durante el fit del train.


In [ ]:
_required_audit_cols = ["label_brand_std", "RepackOrderID", "lot_id"]
_missing_audit_cols = [c for c in _required_audit_cols if c not in model_df.columns]
if _missing_audit_cols:
    raise RuntimeError(
        f"Faltan columnas para brand audit: {_missing_audit_cols}. "
        "Ejecuta el notebook desde el inicio."
    )

brand_audit = (
    model_df.groupby('label_brand_std', dropna=False)
    .agg(
        n_workorders=('RepackOrderID', 'nunique'),
        n_lots=('lot_id', 'nunique'),
        loss_mean=('loss_calc', 'mean'),
        loss_median=('loss_calc', 'median')
    )
    .sort_values(['n_lots', 'n_workorders'], ascending=False)
)

print("Brands en dataset analítico:", model_df['label_brand_std'].nunique(dropna=True))
print("Brands con >= 2 lotes:", int((brand_audit['n_lots'] >= 2).sum()))
display(brand_audit.head(20).round(3))


## Features

El modelo utiliza únicamente información disponible al momento de la primera inspección o derivada de ella.

### Contexto precargado / scan
- `label_brand`
- `country_origin`
- `size_norm`
- mes de inspección / recepción cuando corresponda

### QC de primera muestra
- count / sample size
- temperatura
- defectos registrados manualmente que estén disponibles en ese momento

### Vision AI
- `oleocellosis_pct` como feature puente principal

Se excluyen `Level_Flag`, `open_rating`, `color_rating`, variables post-repack y componentes directos del target.

`label_brand` se incorpora como variable categórica porque representa información real de procedencia/grupo comercial disponible antes de Vision. Su utilidad debe revisarse con validation para evitar sobreajuste.


In [ ]:
NUMERIC_FEATURES = [
    'inspection_month',
    'sample_count_total',
    'n_pallets',
    'avg_count_per_box_mean',
    'avg_count_per_box_max',
    'pulp_temp_mean',
    'pulp_temp_max',
    'oleocellosis_pct',
    'oleocellosis_pct_max_pallet',
    'styler_pct',
    'styler_pct_max_pallet',
    'skin_breakdown_pct',
    'skin_breakdown_pct_max_pallet',
    'decay_pct',
    'decay_pct_max_pallet',
    'mold_pct',
    'mold_pct_max_pallet',
    'scars_pct',
    'scars_pct_max_pallet',
    'blanching_pct',
    'blanching_pct_max_pallet',
    'yellow_color_pct',
    'yellow_color_pct_max_pallet',
]

CATEGORICAL_FEATURES = ['label_brand_std', 'country_origin_std', 'size_norm']

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = 'loss_calc'
GROUP = 'lot_id'

X = model_df[FEATURES].copy()
y = model_df[TARGET].astype(float).copy()
groups = model_df[GROUP].astype(str).copy()

print("Filas:", len(X))
print("Lotes/grupos:", groups.nunique())
print("Features:", len(FEATURES))


### Validación previa al entrenamiento

Esta celda confirma que el EDA ya se ejecutó y que las features estandarizadas existen antes del split.


In [ ]:
_required_model_columns = [
    "label_brand_std",
    "country_origin_std",
    "size_norm",
]

_missing_model_columns = [
    c for c in _required_model_columns
    if c not in model_df.columns
]

if _missing_model_columns:
    raise RuntimeError(
        "Faltan columnas estandarizadas en model_df: "
        f"{_missing_model_columns}. Ejecuta el notebook desde el inicio."
    )

print("✓ Dataset listo para modelado.")
print("Categorical features:", CATEGORICAL_FEATURES)
print("model_df shape:", model_df.shape)


In [ ]:
# 5. SPLIT TRAIN / VALIDATION / TEST SIN LEAKAGE POR LOTE
#
# El TEST queda congelado.
# Huber vs Gradient Boosting se selecciona SOLO con VALIDATION.

if groups.nunique() < 6:
    raise ValueError("Muy pocos lotes únicos para train/validation/test independiente.")

# Separar TEST
gss_test = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEMILLA)
trainval_idx, test_idx = next(gss_test.split(X, y, groups=groups))

X_trainval = X.iloc[trainval_idx].copy()
y_trainval = y.iloc[trainval_idx].copy()
groups_trainval = groups.iloc[trainval_idx].copy()

X_test = X.iloc[test_idx].copy()
y_test = y.iloc[test_idx].copy()
groups_test = groups.iloc[test_idx].copy()

# Separar VALIDATION dentro de train+validation
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEMILLA + 1)
train_rel_idx, val_rel_idx = next(
    gss_val.split(X_trainval, y_trainval, groups=groups_trainval)
)

X_train = X_trainval.iloc[train_rel_idx].copy()
y_train = y_trainval.iloc[train_rel_idx].copy()
groups_train = groups_trainval.iloc[train_rel_idx].copy()

X_val = X_trainval.iloc[val_rel_idx].copy()
y_val = y_trainval.iloc[val_rel_idx].copy()
groups_val = groups_trainval.iloc[val_rel_idx].copy()

assert set(groups_train).isdisjoint(set(groups_val))
assert set(groups_train).isdisjoint(set(groups_test))
assert set(groups_val).isdisjoint(set(groups_test))

print(f"Train: {len(X_train)} WO | {groups_train.nunique()} lotes")
print(f"Val  : {len(X_val)} WO | {groups_val.nunique()} lotes")
print(f"Test : {len(X_test)} WO | {groups_test.nunique()} lotes")
print("TEST congelado hasta la evaluación final.")


In [ ]:
# 6. BASELINE + MODELOS
#
# Selección de modelo: VALIDATION
# Evaluación final: TEST

numeric_pipe_scaled = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

numeric_pipe_tree = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=2, sparse_output=False)),
])

def make_huber():
    prep = ColumnTransformer([
        ('num', numeric_pipe_scaled, NUMERIC_FEATURES),
        ('cat', categorical_pipe, CATEGORICAL_FEATURES),
    ])
    return Pipeline([
        ('prep', prep),
        ('model', HuberRegressor(epsilon=1.35, alpha=0.01, max_iter=1000)),
    ])

def make_gbr():
    prep = ColumnTransformer([
        ('num', numeric_pipe_tree, NUMERIC_FEATURES),
        ('cat', categorical_pipe, CATEGORICAL_FEATURES),
    ])
    return Pipeline([
        ('prep', prep),
        ('model', GradientBoostingRegressor(
            random_state=SEMILLA,
            n_estimators=150,
            learning_rate=0.03,
            max_depth=2,
            loss='huber'
        )),
    ])

# Baseline para selección: mediana SOLO de TRAIN
baseline_train_value = float(np.median(y_train))
pred_val_baseline = np.repeat(baseline_train_value, len(y_val))

candidate_models = {
    'Huber Regression': make_huber(),
    'Gradient Boosting v0': make_gbr(),
}

val_rows = []

for name, model in candidate_models.items():
    model.fit(X_train, y_train)
    pred_val = model.predict(X_val)
    val_rows.append({
        'Modelo': name,
        'MAE_val': mean_absolute_error(y_val, pred_val),
        'RMSE_val': mean_squared_error(y_val, pred_val) ** 0.5,
    })

val_baseline_mae = mean_absolute_error(y_val, pred_val_baseline)

val_results = pd.DataFrame(val_rows).sort_values('MAE_val')
val_results['Mejora_MAE_vs_baseline_val_%'] = (
    (val_baseline_mae - val_results['MAE_val']) / val_baseline_mae * 100
)

print("VALIDATION")
print("Baseline mediana TRAIN:", round(baseline_train_value, 3))
display(val_results.round(3))

selected_model_name = val_results.iloc[0]['Modelo']
print("Modelo seleccionado usando SOLO validation:", selected_model_name)

# Reentrenar seleccionado con TRAIN + VALIDATION
X_dev = pd.concat([X_train, X_val], axis=0)
y_dev = pd.concat([y_train, y_val], axis=0)

best_model = make_huber() if selected_model_name == 'Huber Regression' else make_gbr()
best_model.fit(X_dev, y_dev)

# Baseline final calculado sin tocar test
baseline_value = float(np.median(y_dev))
pred_baseline = np.repeat(baseline_value, len(y_test))

# Test: evaluamos los modelos predefinidos sin cambiar la selección
huber_test = make_huber()
gbr_test = make_gbr()
huber_test.fit(X_dev, y_dev)
gbr_test.fit(X_dev, y_dev)

pred_huber = huber_test.predict(X_test)
pred_gbr = gbr_test.predict(X_test)
pred_best = best_model.predict(X_test)

print("TEST listo para evaluación final.")


In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'MedAE': median_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan,
    }

def grouped_bootstrap_mae(y_true, y_pred, group_ids, n_boot=1000, seed=42):
    df = pd.DataFrame({
        'y': np.asarray(y_true),
        'pred': np.asarray(y_pred),
        'group': np.asarray(group_ids)
    }).reset_index(drop=True)

    unique_groups = df['group'].unique()
    if len(unique_groups) < 2:
        return (np.nan, np.nan)

    rng_local = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        sampled_groups = rng_local.choice(unique_groups, size=len(unique_groups), replace=True)
        pieces = [df[df['group'].eq(g)] for g in sampled_groups]
        b = pd.concat(pieces, ignore_index=True)
        vals.append(mean_absolute_error(b['y'], b['pred']))

    return np.percentile(vals, [2.5, 97.5])

baseline_mae = mean_absolute_error(y_test, pred_baseline)
results = []

for name, pred in [
    ('Baseline mediana', pred_baseline),
    ('Huber Regression', pred_huber),
    ('Gradient Boosting v0', pred_gbr),
]:
    m = regression_metrics(y_test, pred)
    lo, hi = grouped_bootstrap_mae(y_test, pred, groups_test, n_boot=1000, seed=SEMILLA)
    m.update({
        'Modelo': name,
        'MAE_IC95_low': lo,
        'MAE_IC95_high': hi,
        'Mejora_MAE_vs_baseline_%': (baseline_mae - m['MAE']) / baseline_mae * 100
    })
    results.append(m)

results_df = pd.DataFrame(results)[[
    'Modelo','MAE','MAE_IC95_low','MAE_IC95_high',
    'RMSE','MedAE','R2','Mejora_MAE_vs_baseline_%'
]].sort_values('MAE')

display(results_df.round(3))
print('Modelo seleccionado previamente en validation:', selected_model_name)
print('El test NO se usa para cambiar esta selección.')


In [ ]:
comparison = pd.DataFrame({
    'Real': y_test.to_numpy(),
    'Pred_Selected': pred_best,
    'Lot': groups_test.to_numpy()
})

plt.figure(figsize=(7, 5))
plt.scatter(comparison['Real'], comparison['Pred_Selected'])
mn = min(comparison['Real'].min(), comparison['Pred_Selected'].min())
mx = max(comparison['Real'].max(), comparison['Pred_Selected'].max())
plt.plot([mn, mx], [mn, mx])
plt.xlabel('Loss % real')
plt.ylabel('Loss % predicho')
plt.title(f'{selected_model_name} · Real vs predicho')
plt.tight_layout()
plt.show()

errors = comparison['Real'] - comparison['Pred_Selected']
plt.figure(figsize=(7, 4))
plt.hist(errors, bins=15)
plt.xlabel('Error = real - predicho')
plt.ylabel('Frecuencia')
plt.title(f'Distribución de errores · {selected_model_name}')
plt.tight_layout()
plt.show()


In [ ]:
# 9. ANÁLISIS POR SEGMENTOS DEL MODELO SELECCIONADO

# Construir primero el dataframe de evaluación del test.
test_eval = model_df.iloc[test_idx].copy()
test_eval["pred_selected"] = pred_best
test_eval["abs_error"] = np.abs(
    test_eval["loss_calc"] - test_eval["pred_selected"]
)

# Validar DESPUÉS de crear test_eval.
_required_segment_cols = [
    "country_origin_std",
    "label_brand_std",
    "loss_calc",
    "oleocellosis_pct",
]

_missing_segment_cols = [
    c for c in _required_segment_cols
    if c not in test_eval.columns
]

if _missing_segment_cols:
    raise RuntimeError(
        f"Faltan columnas para análisis por segmentos: {_missing_segment_cols}. "
        "Revisa la construcción de model_df."
    )

# Segmento por país de origen estandarizado
seg_origin = (
    test_eval.groupby("country_origin_std", dropna=False)
    .agg(
        n=("loss_calc", "size"),
        MAE=("abs_error", "mean"),
        loss_real_mean=("loss_calc", "mean"),
        loss_pred_mean=("pred_selected", "mean"),
    )
    .sort_values("MAE", ascending=False)
)

print("Por país de origen")
display(seg_origin.round(3))

# Segmento por nivel de oleocelosis
oleo_cut = X_dev["oleocellosis_pct"].median()

test_eval["oleo_segment"] = np.where(
    test_eval["oleocellosis_pct"] > oleo_cut,
    "Oleocelosis > mediana DEV",
    "Oleocelosis <= mediana DEV",
)

seg_oleo = (
    test_eval.groupby("oleo_segment")
    .agg(
        n=("loss_calc", "size"),
        MAE=("abs_error", "mean"),
        loss_real_mean=("loss_calc", "mean"),
        loss_pred_mean=("pred_selected", "mean"),
    )
)

print("Por nivel de oleocelosis")
display(seg_oleo.round(3))

# Segmento adicional por brand estandarizado
seg_brand = (
    test_eval.groupby("label_brand_std", dropna=False)
    .agg(
        n=("loss_calc", "size"),
        MAE=("abs_error", "mean"),
        loss_real_mean=("loss_calc", "mean"),
        loss_pred_mean=("pred_selected", "mean"),
    )
    .sort_values(["n", "MAE"], ascending=[False, False])
)

print("Por label_brand estandarizado")
display(seg_brand.round(3))


## Contrato de integración: Scan + QC + Vision

### A. `scan_context`
Se obtiene automáticamente desde `reception_manifest` al leer `scan_code`.

Ejemplo:
`SF-5815-5638` → lote 5815 · Paramount · Mexico · pallet 5638 · size 200.

Campos de contexto:
- `lot_id`
- `label_brand`
- `country_origin`
- `pallet_id`
- `size_declared`
- `expected_count_per_box`
- `packing_date` si está disponible
- `qty_received_weight` si está disponible

**No provienen de Vision.**

### B. `qc_context`
Información generada durante la primera inspección:
- temperatura
- count real / sample count
- otros defectos manuales disponibles

### C. `vision_output`
Solo información derivada de la imagen:
- `image_id`
- `total_fruits`
- `oleocellosis_count`
- `oleocellosis_pct`
- `mean_confidence` si está disponible

Si Vision entrega `total_fruits` + `oleocellosis_count`, el notebook puede calcular automáticamente `oleocellosis_pct`.

Para el modelo histórico v1.3, la feature visual puente principal sigue siendo `oleocellosis_pct`.


In [ ]:
# 10.1 Modelo seleccionado para integración

best_ml_name = selected_model_name

print("Modelo listo para integración:", best_ml_name)
print("Salida principal: predicted_loss_pct")
print("No se generan automáticamente bandas LOW/MEDIUM/HIGH.")
print("Standard / Expand / Escalate pertenece al Decision Layer.")


In [ ]:
SCAN_REQUIRED_FIELDS = [
    "scan_code", "lot_id", "label_brand", "country_origin",
    "pallet_id", "size_declared", "expected_count_per_box"
]

def get_scan_context(scan_code, manifest=None):
    """
    Recupera el contexto ya conocido antes de Vision a partir del scan.
    """
    if manifest is None:
        manifest = reception_manifest

    if manifest is None:
        raise RuntimeError("Reception manifest no disponible.")

    if "scan_code" not in manifest.columns:
        raise KeyError("El manifest no contiene scan_code.")

    matches = manifest.loc[manifest["scan_code"].astype(str) == str(scan_code)]

    if len(matches) == 0:
        raise KeyError(f"scan_code no encontrado: {scan_code}")
    if len(matches) > 1:
        raise ValueError(f"scan_code duplicado en manifest: {scan_code}")

    row = matches.iloc[0].to_dict()

    missing = [f for f in SCAN_REQUIRED_FIELDS if str(row.get(f, "")).strip() == ""]
    if missing:
        raise ValueError(f"Faltan campos requeridos en manifest para {scan_code}: {missing}")

    size_declared = str(row.get("size_declared", "")).strip()

    return {
        "scan_code": row.get("scan_code"),
        "shipment_id": row.get("shipment_id"),
        "lot_id": row.get("lot_id"),
        "label_brand": row.get("label_brand"),
        "label_brand_std": BRAND_ALIASES.get(
            normalize_brand_key(row.get("label_brand")),
            normalize_brand_key(row.get("label_brand"))
        ),
        "country_origin": row.get("country_origin"),
        "country_origin_std": normalize_text_basic(row.get("country_origin")),
        "pallet_id": row.get("pallet_id"),
        "packing_date": row.get("packing_date"),
        "size_declared": size_declared,
        # Mapeo al nombre usado por el modelo histórico
        "size_norm": size_declared,
        "expected_count_per_box": pd.to_numeric(
            pd.Series([row.get("expected_count_per_box")]), errors="coerce"
        ).iloc[0],
        "qty_received_weight": pd.to_numeric(
            pd.Series([row.get("qty_received_weight")]), errors="coerce"
        ).iloc[0],
        "boxes_received": pd.to_numeric(
            pd.Series([row.get("boxes_received")]), errors="coerce"
        ).iloc[0],
        "data_origin": row.get("data_origin"),
    }


def normalize_vision_output(vision_output):
    """
    Valida y normaliza la salida de Vision.
    Calcula oleocellosis_pct si Vision entrega count + total.
    """
    v = dict(vision_output or {})

    total = pd.to_numeric(pd.Series([v.get("total_fruits")]), errors="coerce").iloc[0]
    affected = pd.to_numeric(pd.Series([v.get("oleocellosis_count")]), errors="coerce").iloc[0]
    pct = pd.to_numeric(pd.Series([v.get("oleocellosis_pct")]), errors="coerce").iloc[0]

    if pd.isna(pct):
        if pd.notna(total) and total > 0 and pd.notna(affected):
            pct = float(affected) / float(total) * 100.0
        else:
            raise ValueError(
                "Vision debe entregar oleocellosis_pct o "
                "total_fruits + oleocellosis_count."
            )

    if pd.notna(total) and total < 0:
        raise ValueError("total_fruits no puede ser negativo.")
    if pd.notna(affected) and affected < 0:
        raise ValueError("oleocellosis_count no puede ser negativo.")
    if pd.notna(total) and pd.notna(affected) and affected > total:
        raise ValueError("oleocellosis_count no puede ser mayor que total_fruits.")

    v["total_fruits"] = None if pd.isna(total) else int(total)
    v["oleocellosis_count"] = None if pd.isna(affected) else int(affected)
    v["oleocellosis_pct"] = float(pct)
    return v


In [ ]:
def predict_lot(scan_context, qc_context, vision_output):
    """
    Predictive Loss v1.3.

    scan_context:
        Contexto recuperado desde reception_manifest mediante scan_code.

    qc_context:
        Variables de la primera inspección no producidas por Vision.

    vision_output:
        Resultados derivados de imagen.

    Returns:
        Señal predictiva + trazabilidad para Decision Layer.
    """
    row = {}
    row.update(scan_context or {})
    row.update(qc_context or {})

    vision = normalize_vision_output(vision_output)
    row["oleocellosis_pct"] = vision["oleocellosis_pct"]

    # Solo se pasan al modelo las features con las que fue entrenado.
    x_new = pd.DataFrame([{c: row.get(c, np.nan) for c in FEATURES}])
    pred = float(best_model.predict(x_new)[0])

    trace = {
        "scan_code": (scan_context or {}).get("scan_code"),
        "shipment_id": (scan_context or {}).get("shipment_id"),
        "lot_id": (scan_context or {}).get("lot_id"),
        "label_brand": (scan_context or {}).get("label_brand"),
        "label_brand_std": (scan_context or {}).get("label_brand_std"),
        "country_origin": (scan_context or {}).get("country_origin"),
        "country_origin_std": (scan_context or {}).get("country_origin_std"),
        "pallet_id": (scan_context or {}).get("pallet_id"),
        "size_declared": (scan_context or {}).get("size_declared"),
        "expected_count_per_box": (scan_context or {}).get("expected_count_per_box"),
        "image_id": vision.get("image_id"),
        "total_fruits": vision.get("total_fruits"),
        "oleocellosis_count": vision.get("oleocellosis_count"),
        "oleocellosis_pct": vision.get("oleocellosis_pct"),
        "vision_confidence": vision.get("mean_confidence"),
    }

    return {
        "predicted_loss_pct": pred,
        "model_name": best_ml_name,
        "trace": trace,
        "decision": None,
        "decision_note": (
            "Standard / Expand / Escalate se resuelve en Decision Layer "
            "combinando Rules Engine + Vision AI + señal predictiva."
        ),
        "model_note": (
            "Predictive Loss v1.3 exploratorio; no equivale aún a Hidden Quality Risk."
        ),
    }


In [ ]:
# DEMO DE INTEGRACIÓN
# Simula: escanear → recuperar metadata → recibir Vision → ejecutar Predictive.

if reception_manifest is None:
    raise RuntimeError("Sube SmartFruit_reception_manifest.csv para ejecutar esta demo.")

scan_code_demo = reception_manifest.iloc[0]["scan_code"]
scan_context_demo = get_scan_context(scan_code_demo)

# Estas variables representan lo que QC captura durante la primera muestra.
# Para la demo técnica usamos valores de una fila del test histórico solo para completar
# el contrato; en el prototipo real vendrán de la sesión de inspección.
example_idx = X_test.index[0]
example = model_df.loc[example_idx]

qc_context_demo = {
    c: example.get(c)
    for c in NUMERIC_FEATURES
    if c not in ["inspection_month", "oleocellosis_pct"]
}

# Vision NO devuelve metadata del lote.
vision_output_demo = {
    "image_id": "IMG_DEMO_001",
    "total_fruits": 120,
    "oleocellosis_count": 9,
    # oleocellosis_pct puede omitirse; normalize_vision_output lo calcula = 7.5
    "mean_confidence": 0.90,
}

demo_output = predict_lot(
    scan_context=scan_context_demo,
    qc_context=qc_context_demo,
    vision_output=vision_output_demo,
)

print("SCAN:", scan_code_demo)
print("Contexto recuperado:")
print(scan_context_demo)
print("\nSalida Predictive:")
print(demo_output)


In [ ]:
artifact_dir = MODEL_OUTPUT_DIR
artifact_dir.mkdir(exist_ok=True)

model_path = artifact_dir / "predictive_loss_v1_3.joblib"
meta_path = artifact_dir / "predictive_loss_v1_3_metadata.json"

joblib.dump(best_model, model_path)

metadata = {
    'model_name': best_ml_name,
    'selection_method': 'grouped validation; test frozen',
    'target': TARGET,
    'unit': 'single-lot Work Order linked to first QC',
    'features': FEATURES,
    'numeric_features': NUMERIC_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'baseline_dev_median': baseline_value,
    'test_lots': sorted(groups_test.unique().tolist()),
    'excluded_negative_loss': EXCLUIR_LOSS_NEGATIVO,
    'vision_bridge_feature': 'oleocellosis_pct',
    'input_contract': {
        'reception_manifest': 'SmartFruit_reception_manifest.csv',
        'scan_key': 'scan_code',
        'scan_context': ['lot_id', 'label_brand', 'label_brand_std', 'country_origin', 'country_origin_std', 'size_norm', 'inspection_month'],
        'vision_output_required_for_model': ['oleocellosis_pct'],
        'vision_output_traceability': ['image_id', 'total_fruits', 'oleocellosis_count', 'mean_confidence']
    },
    'output_contract': {
        'predicted_loss_pct': 'numeric signal for Decision Layer',
        'decision': 'not produced by this model'
    },
    'warning': (
        'Predictive Loss v1.3 is exploratory. Vision AI oleocellosis estimates '
        'must be validated against historical/manual measurements. '
        'This model does not directly produce Standard/Expand/Escalate and '
        'does not yet represent Hidden Quality Risk.'
    )
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Modelo guardado en:", model_path)
print("Metadata guardada en:", meta_path)


## Criterio de lectura

- **Baseline:** mediana del loss% calculada sin tocar el test.
- **Modelo simple:** Huber Regression.
- **Modelo v0:** Gradient Boosting.
- **Selección:** validation agrupada por lote.
- **Evaluación final:** test congelado.
- **Métrica principal:** MAE.

### Rutas oficiales

```text
MyDrive/SMARTFRUIT/data/reportes.zip
MyDrive/SMARTFRUIT/data/Weight IN - OUT by lot - item.xlsx
MyDrive/SMARTFRUIT/integration/SmartFruit_reception_manifest.csv
MyDrive/SMARTFRUIT/vision/
MyDrive/SMARTFRUIT/models/
```

### Flujo de integración

`scan_code`
→ `get_scan_context()`
→ metadata de recepción
→ `qc_context`
→ `vision_output`
→ `predict_lot()`
→ `predicted_loss_pct`

Vision no infiere brand, país, lote, pallet ni size; esa información ya existe desde el scan.

`predicted_loss_pct` sigue siendo una señal para el Decision Layer, no una decisión final.
